In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier

In [2]:
# ==========================================
# INPUT CSV PATH
# ==========================================

csv_path = "data/Australia_ODI_Cricketers_with_Role.csv"   # change this to your uploaded CSV path

In [3]:
# ==========================================
# LOAD CSV
# ==========================================

df_raw = pd.read_csv(csv_path)

print("Dataset loaded successfully.")
print("Shape:", df_raw.shape)
print("\nColumns:")
print(df_raw.columns.tolist())

df_raw.head()

Dataset loaded successfully.
Shape: (253, 21)

Columns:
['Cap', 'Name', 'Career', 'Mat', 'Inn', 'NO', 'Runs', 'HS', 'Avg', '100/50', 'Balls', 'Mdn', 'Runs.1', 'Wkt', 'BB', 'Avg.1', '5wI', 'Ca', 'St', 'Ref', 'Role']


,Cap,Name,Career,Mat,Inn,NO,Runs,HS,Avg,100/50,...,Mdn,Runs.1,Wkt,BB,Avg.1,5wI,Ca,St,Ref,Role
0,1,Greg Chappell,1971–1983,74.0,72.0,14.0,"2,331",138*,40.18,Mar-14,...,41.0,"2,097",72.0,May-15,29.12,2.0,23.0,0.0,[4],Batsman
1,2,Ian Chappell,1971–1980,16.0,16.0,2.0,673,86,48.07,0/8,...,1.0,23,2.0,Feb-14,11.50,0.0,5.0,0.0,[5],Batsman
2,3,Alan Connolly,1971,1.0,0.0,0.0,0,NaN,NaN,0/0,...,0.0,62,0.0,NaN,NaN,0.0,0.0,0.0,[6],Batsman
3,4,Bill Lawry,1971,1.0,1.0,0.0,27,27,27.00,0/0,...,0.0,0,0.0,NaN,NaN,0.0,1.0,0.0,[7],Batsman
4,5,Graham McKenzie,1971,1.0,0.0,0.0,0,NaN,NaN,0/0,...,0.0,22,2.0,Feb-22,11.00,0.0,1.0,0.0,[8],Batsman


In [4]:
# ==========================================
# STANDARDIZE COLUMN NAMES AND STRUCTURE
# ==========================================

def load_and_standardize_csv(csv_path):
    df = pd.read_csv(csv_path)

    # Replace weird dash values with NaN
    df.replace('—', np.nan, inplace=True)

    # --------------------------------------
    # STANDARD TARGET COLUMNS
    # --------------------------------------
    required_columns = [
        'Name',
        'First',
        'Last',
        'Mat',
        'Inn',
        'NO.1',
        'Runs',
        'HS',
        'Avg',
        'Balls',
        'Mdn',
        'Runs_1',
        'Wkt',
        'BBM',
        'Avg_2',
        'Ca',
        'St',
        'Role'
    ]

    # If some required columns are missing, create them
    for col in required_columns:
        if col not in df.columns:
            if col in ['Name', 'HS', 'BBM', 'Role']:
                df[col] = ""
            else:
                df[col] = 0

    # --------------------------------------
    # CLEAN NUMERIC COLUMNS
    # --------------------------------------
    numeric_cols = [
        'First',
        'Last',
        'Mat',
        'Inn',
        'NO.1',
        'Runs',
        'Avg',
        'Balls',
        'Mdn',
        'Runs_1',
        'Wkt',
        'Avg_2',
        'Ca',
        'St'
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df[numeric_cols] = df[numeric_cols].fillna(0)

    # Clean text columns
    df['Name'] = df['Name'].astype(str).str.strip()
    df['Role'] = df['Role'].astype(str).str.strip()

    # If Selected does not exist, create default 0
    if 'Selected' not in df.columns:
        df['Selected'] = 0

    df['Selected'] = pd.to_numeric(df['Selected'], errors='coerce').fillna(0).astype(int)

    return df

In [5]:


def create_features(df):
    df = df.copy()

    # ------------------------------------------
    # 1. Define numeric cricket-stat columns
    # ------------------------------------------
    numeric_cols = [
        'Mat', 'Inn', 'NO.1', 'Runs', 'Avg',
        'Balls', 'Mdn', 'Runs_1', 'Wkt',
        'Avg_2', 'Ca', 'St', 'First', 'Last'
    ]

    # Keep only those that actually exist in the uploaded CSV
    numeric_cols = [col for col in numeric_cols if col in df.columns]

    # Convert them to numeric safely
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Fill NA only in numeric columns
    df[numeric_cols] = df[numeric_cols].fillna(0)

    # ------------------------------------------
    # 2. Avoid division by zero
    # ------------------------------------------
    if 'Inn' in df.columns:
        df['Inn_nonzero'] = df['Inn'].replace(0, 1)
    else:
        df['Inn_nonzero'] = 1

    if 'Balls' in df.columns:
        df['Balls_nonzero'] = df['Balls'].replace(0, 1)
    else:
        df['Balls_nonzero'] = 1

    # ------------------------------------------
    # 3. Create batting features
    # ------------------------------------------
    if 'Runs' in df.columns and 'Balls' in df.columns:
        df['strike_rate'] = (df['Runs'] / df['Balls_nonzero']) * 100
    else:
        df['strike_rate'] = 0

    if 'Runs' in df.columns and 'Inn' in df.columns:
        df['consistency'] = df['Runs'] / df['Inn_nonzero']
    else:
        df['consistency'] = 0

    # ------------------------------------------
    # 4. Create bowling feature
    # ------------------------------------------
    if 'Runs_1' in df.columns and 'Balls' in df.columns:
        df['economy'] = (df['Runs_1'] / df['Balls_nonzero']) * 6
    else:
        df['economy'] = 0

    # ------------------------------------------
    # 5. Clean up infinities
    # ------------------------------------------
    df.replace([np.inf, -np.inf], 0, inplace=True)

    # Fill only newly created numeric feature columns
    feature_cols = ['strike_rate', 'consistency', 'economy']
    df[feature_cols] = df[feature_cols].fillna(0)

    return df

In [6]:
# ==========================================
# ROLE-WISE SCORING FUNCTIONS
# ==========================================

def compute_batsman_score(role_df):
    temp = role_df.copy()
    scaler = MinMaxScaler()

    features = ['Avg', 'strike_rate', 'Runs', 'consistency', 'Mat']
    temp[features] = scaler.fit_transform(temp[features])

    temp['role_score'] = (
        temp['Avg'] * 0.30 +
        temp['strike_rate'] * 0.25 +
        temp['Runs'] * 0.20 +
        temp['consistency'] * 0.15 +
        temp['Mat'] * 0.10
    )

    return temp


def compute_bowler_score(role_df):
    temp = role_df.copy()
    scaler = MinMaxScaler()

    features = ['Wkt', 'Avg_2', 'Mdn', 'economy', 'Mat']
    temp[features] = scaler.fit_transform(temp[features])

    temp['role_score'] = (
        temp['Wkt'] * 0.40 +
        (1 - temp['Avg_2']) * 0.25 +
        temp['Mdn'] * 0.15 +
        (1 - temp['economy']) * 0.10 +
        temp['Mat'] * 0.10
    )

    return temp


def compute_allrounder_score(role_df):
    temp = role_df.copy()
    scaler = MinMaxScaler()

    features = ['Runs', 'Avg', 'strike_rate', 'Wkt', 'Avg_2', 'Mdn', 'Mat']
    temp[features] = scaler.fit_transform(temp[features])

    temp['role_score'] = (
        temp['Runs'] * 0.20 +
        temp['Avg'] * 0.15 +
        temp['strike_rate'] * 0.15 +
        temp['Wkt'] * 0.25 +
        (1 - temp['Avg_2']) * 0.10 +
        temp['Mdn'] * 0.05 +
        temp['Mat'] * 0.10
    )

    return temp


def compute_wicketkeeper_score(role_df):
    temp = role_df.copy()
    scaler = MinMaxScaler()

    features = ['Runs', 'Avg', 'strike_rate', 'Ca', 'St', 'Mat']
    temp[features] = scaler.fit_transform(temp[features])

    temp['role_score'] = (
        temp['Runs'] * 0.25 +
        temp['Avg'] * 0.20 +
        temp['strike_rate'] * 0.20 +
        temp['Ca'] * 0.15 +
        temp['St'] * 0.10 +
        temp['Mat'] * 0.10
    )

    return temp

In [7]:
# ==========================================
# GENERATE ROLE PREDICTIONS
# ==========================================

def generate_role_predictions(df, role_name, recent_only=True):
    role_df = df.copy()

    # --------------------------------------
    # Filter by role
    # --------------------------------------
    role_df = role_df[role_df['Role'].str.lower() == role_name.lower()].copy()

    # --------------------------------------
    # Recent players only
    # --------------------------------------
    if recent_only and 'Last' in role_df.columns:
        role_df = role_df[role_df['Last'] >= 2020].copy()

    # --------------------------------------
    # Minimum match filters to avoid tiny-sample weirdness
    # --------------------------------------
    if role_name.lower() == "batsman":
        role_df = role_df[role_df['Mat'] >= 5]

    elif role_name.lower() == "bowler":
        role_df = role_df[role_df['Mat'] >= 5]

    elif role_name.lower() == "allrounder":
        role_df = role_df[role_df['Mat'] >= 5]

    elif role_name.lower() == "wicketkeeper":
        role_df = role_df[role_df['Mat'] >= 3]

    if len(role_df) == 0:
        print(f"No players found for role: {role_name}")
        return pd.DataFrame()

    # --------------------------------------
    # ROLE-BASED SCORE
    # --------------------------------------
    if role_name.lower() == "batsman":
        role_df = compute_batsman_score(role_df)

    elif role_name.lower() == "bowler":
        role_df = compute_bowler_score(role_df)

    elif role_name.lower() == "allrounder":
        role_df = compute_allrounder_score(role_df)

    elif role_name.lower() == "wicketkeeper":
        role_df = compute_wicketkeeper_score(role_df)

    else:
        print(f"Unknown role: {role_name}")
        return pd.DataFrame()

    # --------------------------------------
    # ML PROBABILITY (ONLY IF POSSIBLE)
    # --------------------------------------
    role_df['selection_probability'] = 0.0

    # Use ML only if Selected column has both 0 and 1
    if 'Selected' in role_df.columns and role_df['Selected'].nunique() >= 2:
        ml_features = [
            'Mat', 'Inn', 'Runs', 'Avg',
            'Wkt', 'Avg_2',
            'strike_rate', 'consistency', 'economy'
        ]

        # Ensure columns exist
        for col in ml_features:
            if col not in role_df.columns:
                role_df[col] = 0

        X = role_df[ml_features].fillna(0)
        y = role_df['Selected']

        scaler = MinMaxScaler()
        X_scaled = scaler.fit_transform(X)

        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42
        )
        model.fit(X_scaled, y)

        role_df['selection_probability'] = model.predict_proba(X_scaled)[:, 1]

        # --------------------------------------
        # COMBINE ML + ROLE SCORE
        # --------------------------------------
        role_df['final_score'] = (
            role_df['selection_probability'] * 0.60 +
            role_df['role_score'] * 0.40
        )

    else:
        # If ML not possible, use role_score only
        role_df['selection_probability'] = role_df['role_score']
        role_df['final_score'] = role_df['role_score']

    # --------------------------------------
    # SORT FINAL
    # --------------------------------------
    role_df = role_df.sort_values(
        by='final_score',
        ascending=False
    ).reset_index(drop=True)

    return role_df

In [8]:
# ==========================================
# GENERATE FINAL 15-MEMBER SQUAD
# ==========================================

def generate_final_squad(
    best_batsmen,
    best_bowlers,
    best_allrounders,
    best_wicketkeepers
):
    # --------------------------------------
    # MAIN PLAYING XI
    # --------------------------------------
    main_batsmen = best_batsmen.head(4)
    main_allrounders = best_allrounders.head(2)
    main_wicketkeeper = best_wicketkeepers.head(1)
    main_bowlers = best_bowlers.head(4)

    playing_xi = pd.concat([
        main_batsmen,
        main_allrounders,
        main_wicketkeeper,
        main_bowlers
    ], ignore_index=True)

    # --------------------------------------
    # RESERVES
    # --------------------------------------
    reserve_batsman = best_batsmen.iloc[4:5]
    reserve_wicketkeeper = best_wicketkeepers.iloc[1:2]
    reserve_allrounder = best_allrounders.iloc[2:3]
    reserve_bowler = best_bowlers.iloc[4:5]

    reserves = pd.concat([
        reserve_batsman,
        reserve_wicketkeeper,
        reserve_allrounder,
        reserve_bowler
    ], ignore_index=True)

    # --------------------------------------
    # FINAL SQUAD = XI + reserves
    # --------------------------------------
    final_squad = pd.concat([
        playing_xi,
        reserves
    ], ignore_index=True)

    # Remove duplicates if any player appears twice
    final_squad = final_squad.drop_duplicates(subset='Name').reset_index(drop=True)

    # Keep only required display columns if they exist
    display_cols = ['Name', 'Role', 'selection_probability']
    existing_cols = [col for col in display_cols if col in final_squad.columns]

    return final_squad[existing_cols]

In [9]:
# ==========================================
# MASTER FUNCTION: GENERATE SQUAD FROM CSV
# ==========================================

def generate_squad_from_csv(csv_path):
    # Load and standardize
    df = load_and_standardize_csv(csv_path)

    # Create features
    df = create_features(df)

    # Generate role-wise squads
    best_batsmen = generate_role_predictions(df, "Batsman")
    best_bowlers = generate_role_predictions(df, "Bowler")
    best_allrounders = generate_role_predictions(df, "AllRounder")
    best_wicketkeepers = generate_role_predictions(df, "Wicketkeeper")

    # Final squad
    final_squad = generate_final_squad(
        best_batsmen,
        best_bowlers,
        best_allrounders,
        best_wicketkeepers
    )

    return {
        "cleaned_df": df,
        "best_batsmen": best_batsmen,
        "best_bowlers": best_bowlers,
        "best_allrounders": best_allrounders,
        "best_wicketkeepers": best_wicketkeepers,
        "final_squad": final_squad
    }

In [10]:
# ==========================================
# RUN THE PIPELINE
# ==========================================

results = generate_squad_from_csv(csv_path)

best_batsmen = results["best_batsmen"]
best_bowlers = results["best_bowlers"]
best_allrounders = results["best_allrounders"]
best_wicketkeepers = results["best_wicketkeepers"]
final_squad = results["final_squad"]

No players found for role: Batsman
No players found for role: Bowler
No players found for role: AllRounder
No players found for role: Wicketkeeper


In [72]:
print("BEST BATSMEN")
best_batsmen[['Name', 'Role', 'selection_probability']].head(10)

BEST BATSMEN


KeyError: "None of [Index(['Name', 'Role', 'selection_probability'], dtype='str')] are in the [columns]"

In [58]:
print("BEST BOWLERS")
best_bowlers[['Name', 'Role', 'selection_probability']].head(10)

BEST BOWLERS


KeyError: "None of [Index(['Name', 'Role', 'selection_probability'], dtype='str')] are in the [columns]"

In [59]:
print("BEST ALL-ROUNDERS")
best_allrounders[['Name', 'Role', 'selection_probability']].head(10)

BEST ALL-ROUNDERS


KeyError: "None of [Index(['Name', 'Role', 'selection_probability'], dtype='str')] are in the [columns]"

In [60]:
print("BEST WICKETKEEPERS")
best_wicketkeepers[['Name', 'Role', 'selection_probability']].head(10)

BEST WICKETKEEPERS


KeyError: "None of [Index(['Name', 'Role', 'selection_probability'], dtype='str')] are in the [columns]"

In [61]:
print("FINAL 15-MEMBER SQUAD")
final_squad

FINAL 15-MEMBER SQUAD


""


In [73]:
df_test = load_and_standardize_csv(csv_path)

print("Columns:")
print(df_test.columns.tolist())

print("\nUnique Role values:")
print(df_test['Role'].unique())

print("\nRole counts:")
print(df_test['Role'].value_counts(dropna=False))

print("\nSample rows:")
print(df_test[['Name', 'Role']].head(20))

Columns:
['Cap', 'Name', 'Career', 'Mat', 'Inn', 'NO', 'Runs', 'HS', 'Avg', '100/50', 'Balls', 'Mdn', 'Runs.1', 'Wkt', 'BB', 'Avg.1', '5wI', 'Ca', 'St', 'Ref', 'Role', 'First', 'Last', 'NO.1', 'Runs_1', 'BBM', 'Avg_2', 'Selected']

Unique Role values:
<StringArray>
['Batsman', 'Wicketkeeper', 'Bowler', 'AllRounder']
Length: 4, dtype: str

Role counts:
Role
Batsman         165
Bowler           54
AllRounder       18
Wicketkeeper     16
Name: count, dtype: int64

Sample rows:
               Name          Role
0     Greg Chappell       Batsman
1      Ian Chappell       Batsman
2     Alan Connolly       Batsman
3        Bill Lawry       Batsman
4   Graham McKenzie       Batsman
5    Ashley Mallett       Batsman
6         Rod Marsh  Wicketkeeper
7       Ian Redpath       Batsman
8   Keith Stackpole       Batsman
9      Alan Thomson       Batsman
10     Doug Walters       Batsman
11     Ross Edwards       Batsman
12    Dennis Lillee        Bowler
13       Bob Massie       Batsman
14     Paul